In [1]:
from imports import *

In [2]:
result_list = glob.glob("stats/pi*")

print("Number of tested combinations: {}".format(len(result_list)))

Number of tested combinations: 10


In [3]:
result_list

['stats\\pi-E-n22-k4.evrp--mtr-0.800000--cor-0.700000--mtm-mix--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n22-k4.evrp--mtr-0.800000--cor-0.700000--mtm-swp--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n22-k4.evrp--mtr-0.800000--cor-0.700000--mtm-inv--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n76-k7.evrp--mtr-0.800000--cor-0.700000--mtm-ins--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n76-k7.evrp--mtr-0.800000--cor-0.700000--mtm-inv--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n22-k4.evrp--mtr-0.900000--cor-0.800000--mtm-mix--stp-1.500000--pop-16.txt',
 'stats\\pi-E-n76-k7.evrp--mtr-0.800000--cor-0.700000--mtm-mix--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n22-k4.evrp--mtr-0.900000--cor-0.800000--mtm-swp--stp-1.500000--pop-16.txt',
 'stats\\pi-E-n22-k4.evrp--mtr-0.800000--cor-0.700000--mtm-ins--stp-1.500000--pop-8.txt',
 'stats\\pi-E-n76-k7.evrp--mtr-0.800000--cor-0.700000--mtm-swp--stp-1.500000--pop-8.txt']

In [ ]:
if len(result_list) > 0:

    params = result_list[0].split("stats/")[-1].split(".txt")[0]
    
    print("\nParameters of the first configuration:")

    for i,kvalue in enumerate(params.split("--")):
        if i !=0:
            
        key,value = kvalue.split("-", 1)

        print(key,value)


mtr 0.800000
cor 0.700000
mtm mix
stp 1.500000
pop 8


In [ ]:
results{}

In [ ]:
for j, result_file in enumerate(result_list):

    # Get only the file name
    file_name = os.path.basename(result_file)

    # Remove .txt
    configuration_name = file_name.replace(".txt", "")

    # Extract instance name
    instance_name = (
        configuration_name
        .split("pi-")[-1]
        .split("--mtr-")[0]
    )

    # Store parameters
    parameters = {}

    for i, item in enumerate(configuration_name.split("--")):

        if i == 0:
            continue

        key, value = item.split("-", 1)

        parameters[key] = value


    # Create result structure
    results[j] = {

        "file_name": configuration_name,

        "instance": instance_name,

        "parameters": parameters,

        "routes": [],

        "r_fit": [],

        "best_route": 0,

        "mean_fit": 0,

        "std_fit": 0,

        "max_fit": 0,

        "min_fit": 0,
    }


    # Open the CURRENT result file
    with open(result_file, "r", encoding="utf-8") as file:

        for raw_line in file:

            # Remove spaces and line breaks
            line = raw_line.strip()

            # Ignore empty lines
            if line == "":
                continue


            # ------------------------------------------------
            # ROUTES
            # ------------------------------------------------

            if line.startswith("route:"):

                route_text = line.split("route:", 1)[1]

                route = [
                    int(value.strip())
                    for value in route_text.split(",")
                    if value.strip() != ""
                ]

                results[j]["routes"].append(route)


            # ------------------------------------------------
            # FITNESS
            # ------------------------------------------------

            else:

                try:

                    fitness = float(line)

                    results[j]["r_fit"].append(fitness)

                except ValueError:

                    # Ignore Mean, Std Dev, Min, Max, etc.
                    pass


    # Convert fitness values to NumPy array
    results[j]["r_fit"] = np.array(
        results[j]["r_fit"],
        dtype=float
    )


    # Check whether fitness values were found
    if len(results[j]["r_fit"]) > 0:

        # Calculate statistics directly from the 20 runs
        results[j]["mean_fit"] = np.mean(
            results[j]["r_fit"]
        )

        results[j]["std_fit"] = np.std(
            results[j]["r_fit"],
            ddof=1
        )

        results[j]["min_fit"] = np.min(
            results[j]["r_fit"]
        )

        results[j]["max_fit"] = np.max(
            results[j]["r_fit"]
        )

        # Index of the best route
        results[j]["best_route"] = np.argmin(
            results[j]["r_fit"]
        )


# ============================================================
# SHOW RESULTS
# ============================================================

print("\nConfigurations read:", len(results))

for j in results:

    print(
        "\nConfiguration:",
        j
    )

    print(
        "Instance:",
        results[j]["instance"]
    )

    print(
        "Number of runs:",
        len(results[j]["r_fit"])
    )

    print(
        "Mean:",
        results[j]["mean_fit"]
    )

    print(
        "Standard deviation:",
        results[j]["std_fit"]
    )

    print(
        "Minimum:",
        results[j]["min_fit"]
    )

    print(
        "Maximum:",
        results[j]["max_fit"]
    )


# ============================================================
# CREATE SUMMARY TABLE
# ============================================================

summary = []


for j, data in results.items():

    summary.append({

        "id": j,

        "instance": data["instance"],

        "mutation_rate": data["parameters"].get(
            "mtr",
            ""
        ),

        "crossover_rate": data["parameters"].get(
            "cor",
            ""
        ),

        "mutation": data["parameters"].get(
            "mtm",
            ""
        ),

        "selection_pressure": data["parameters"].get(
            "stp",
            ""
        ),

        "population": data["parameters"].get(
            "pop",
            ""
        ),

        "mean_fit": data["mean_fit"],

        "std_fit": data["std_fit"],

        "min_fit": data["min_fit"],

        "max_fit": data["max_fit"],
    })


summary_df = pd.DataFrame(summary)


# Sort by instance and mean fitness
summary_df = summary_df.sort_values(
    by=[
        "instance",
        "mean_fit"
    ]
)


print("\n========================================")
print("SUMMARY")
print("========================================")

display(summary_df)


# ============================================================
# FIND BEST CONFIGURATION OF EACH INSTANCE
# ============================================================

best_configuration = {}


for instance in summary_df["instance"].unique():

    # Get only configurations of this instance
    instance_data = summary_df[
        summary_df["instance"] == instance
    ]

    # Find configuration with smallest mean fitness
    best_row = instance_data.loc[
        instance_data["mean_fit"].idxmin()
    ]

    best_id = int(
        best_row["id"]
    )

    best_configuration[instance] = best_id


    print(
        "\n========================================"
    )

    print(
        "Instance:",
        instance
    )

    print(
        "Best configuration:",
        best_id
    )

    print(
        "Mean fitness:",
        results[best_id]["mean_fit"]
    )

    print(
        "Mutation:",
        results[best_id]["parameters"].get(
            "mtm",
            ""
        )
    )

    print(
        "========================================"
    )


# ============================================================
# MANN-WHITNEY TEST
# ============================================================

alpha = 0.05

mann_whitney_results = []


for instance in summary_df["instance"].unique():

    # Best configuration for this instance
    best_id = best_configuration[
        instance
    ]

    # 20 runs of the best configuration
    best_fitness = results[
        best_id
    ]["r_fit"]


    # Get all configurations of this instance
    instance_configurations = summary_df[
        summary_df["instance"] == instance
    ]


    # Number of comparisons
    number_of_comparisons = (
        len(instance_configurations) - 1
    )


    for _, row in instance_configurations.iterrows():

        current_id = int(
            row["id"]
        )

        current_fitness = results[
            current_id
        ]["r_fit"]


        # --------------------------------------------
        # BEST CONFIGURATION ITSELF
        # --------------------------------------------

        if current_id == best_id:

            mann_whitney_results.append({

                "instance": instance,

                "configuration": current_id,

                "best_configuration": best_id,

                "mutation": results[
                    current_id
                ]["parameters"].get(
                    "mtm",
                    ""
                ),

                "mean": np.mean(
                    current_fitness
                ),

                "best_mean": np.mean(
                    best_fitness
                ),

                "U": np.nan,

                "p_value": np.nan,

                "p_adjusted": np.nan,

                "result": "Best configuration"
            })

            continue


        # --------------------------------------------
        # MANN-WHITNEY TEST
        # --------------------------------------------

        statistic, p_value = mannwhitneyu(

            current_fitness,

            best_fitness,

            alternative="two-sided",

            method="auto"
        )


        # --------------------------------------------
        # BONFERRONI CORRECTION
        # --------------------------------------------

        if number_of_comparisons > 0:

            p_adjusted = min(

                p_value *
                number_of_comparisons,

                1.0
            )

        else:

            p_adjusted = p_value


        # --------------------------------------------
        # INTERPRET RESULT
        # --------------------------------------------

        if p_adjusted < alpha:

            # Smaller fitness is better
            if np.median(current_fitness) > np.median(best_fitness):

                conclusion = (
                    "Statistically worse"
                )

            elif np.median(current_fitness) < np.median(best_fitness):

                conclusion = (
                    "Statistically better"
                )

            else:

                conclusion = (
                    "Statistically different"
                )

        else:

            conclusion = (
                "No significant difference"
            )


        # Store result
        mann_whitney_results.append({

            "instance": instance,

            "configuration": current_id,

            "best_configuration": best_id,

            "mutation": results[
                current_id
            ]["parameters"].get(
                "mtm",
                ""
            ),

            "mean": np.mean(
                current_fitness
            ),

            "best_mean": np.mean(
                best_fitness
            ),

            "U": statistic,

            "p_value": p_value,

            "p_adjusted": p_adjusted,

            "result": conclusion
        })


# ============================================================
# CREATE MANN-WHITNEY TABLE
# ============================================================

mann_whitney_df = pd.DataFrame(
    mann_whitney_results
)


print("\n========================================")
print("MANN-WHITNEY TEST")
print("========================================")

display(
    mann_whitney_df
)


# ============================================================
# SHOW RESULTS BY INSTANCE
# ============================================================

for instance in mann_whitney_df["instance"].unique():

    print(
        "\n========================================"
    )

    print(
        "INSTANCE:",
        instance
    )

    print(
        "========================================"
    )

    instance_test = mann_whitney_df[
        mann_whitney_df["instance"] == instance
    ]

    display(
        instance_test
    )

ValueError: can only convert an array of size 1 to a Python scalar

In [ ]:
# Process each instance separately
for instance in summary_df["instance"].unique():

    # --------------------------------------------------------
    # Get configurations from the current instance
    # --------------------------------------------------------

    instance_data = summary_df[
        summary_df["instance"] == instance
    ].copy()


    # --------------------------------------------------------
    # Select the three best configurations
    # Smaller fitness is better
    # --------------------------------------------------------

    top_three = instance_data.nsmallest(
        3,
        "mean_fit"
    )


    print("\n========================================")
    print("TOP 3 CONFIGURATIONS")
    print("Instance:", instance)
    print("========================================")

    display(
        top_three[
            [
                "id",
                "mutation",
                "mean_fit",
                "std_fit"
            ]
        ]
    )


    # --------------------------------------------------------
    # Create convergence figure
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 6)
    )


    # --------------------------------------------------------
    # Process each of the three best configurations
    # --------------------------------------------------------

    for rank, (_, row) in enumerate(
        top_three.iterrows(),
        start=1
    ):

        configuration_id = int(
            row["id"]
        )

        configuration = results[
            configuration_id
        ]


        # Get original result file
        result_file = result_list[
            configuration_id
        ]


        # Get configuration name
        file_name = os.path.basename(
            result_file
        )

        configuration_name = file_name.replace(
            ".txt",
            ""
        )


        # ----------------------------------------------------
        # Find convergence file
        # ----------------------------------------------------

        convergence_file = os.path.join(
            "stats",
            "convergence-" +
            configuration_name +
            ".csv"
        )


        # ----------------------------------------------------
        # Check if convergence file exists
        # ----------------------------------------------------

        if not os.path.exists(
            convergence_file
        ):

            print(
                "Convergence file not found:",
                convergence_file
            )

            continue


        # ----------------------------------------------------
        # Read convergence data
        # ----------------------------------------------------

        convergence_data = pd.read_csv(
            convergence_file
        )


        # ----------------------------------------------------
        # Calculate mean convergence curve
        # across the 20 runs
        # ----------------------------------------------------

        mean_convergence = (
            convergence_data
            .groupby("evaluations")["best"]
            .mean()
            .reset_index()
        )


        # ----------------------------------------------------
        # Plot convergence curve
        # ----------------------------------------------------

        mutation = configuration[
            "parameters"
        ].get(
            "mtm",
            ""
        )


        plt.plot(
            mean_convergence["evaluations"],
            mean_convergence["best"],
            linewidth=2,
            label=(
                f"{rank}º - "
                f"{mutation} "
                f"(mean={configuration['mean_fit']:.2f})"
            )
        )


    # --------------------------------------------------------
    # Configure graph
    # --------------------------------------------------------

    plt.xlabel(
        "Function evaluations"
    )

    plt.ylabel(
        "Best fitness"
    )

    plt.title(
        f"Convergence curves - {instance}"
    )

    plt.legend()

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()


    # --------------------------------------------------------
    # Save graph
    # --------------------------------------------------------

    os.makedirs(
        "convergence_plots",
        exist_ok=True
    )


    output_instance = instance.replace(
        ".evrp",
        ""
    )


    output_file = os.path.join(
        "convergence_plots",
        f"top_3_convergence_{output_instance}.png"
    )


    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )


    # --------------------------------------------------------
    # Show graph
    # --------------------------------------------------------

    plt.show()


    print(
        "Figure saved in:",
        output_file
    )